# 12 — Multimodal Prompt Engineering

## Scenario
Northstar automatically extracts data from invoice images submitted by users. 
However, users often provide contradictory text (e.g., typing "Here is my invoice for $800", but attaching an image that says "$500").

**The Problem:** We cannot trust the user's text, nor can we assume the OCR/Model will always perfectly read the image. If there is a contradiction, the model should *not* guess or compromise; it must flag the uncertainty so we can escalate to a human.

In [ ]:
import os
from PIL import Image, ImageDraw
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

# Initialize the client (requires GEMINI_API_KEY environment variable)
client = genai.Client()
MODEL_ID = 'gemini-2.5-flash'

# --- Setup: Create a synthetic invoice image ---
# We generate a simple image locally so this notebook is self-contained.
img = Image.new('RGB', (300, 200), color = (255, 255, 255))
d = ImageDraw.Draw(img)
d.text((10,10), "NORTHSTAR INVOICE", fill=(0,0,0))
d.text((10,50), "Item: Custom Widget", fill=(0,0,0))
d.text((10,90), "TOTAL DUE: $500.00", fill=(255,0,0))
img.save('synthetic_invoice.png')
print("[SYSTEM] Saved 'synthetic_invoice.png' (shows $500.00)")


## Step 1: The Multimodal Contradiction

We will pass the image (showing $500) AND a user prompt (claiming $800) to the model simultaneously. 
We use Pydantic to force the model to explicitly evaluate the evidence rather than just returning a string.

In [ ]:
class InvoiceExtraction(BaseModel):
    is_contradictory: bool = Field(
        description="Set to True if the amount claimed in the user's text contradicts the amount visible in the image."
    )
    extracted_amount: float | None = Field(
        description="The amount visible in the image. Return null if there is a contradiction."
    )
    reasoning: str = Field(
        description="Explain why you flagged this as contradictory or not."
    )

user_text = "Hi, please process my attached invoice for $800.00. Thanks!"
invoice_image = Image.open('synthetic_invoice.png')

prompt = f"""You are an automated invoice processor.\nCarefully review the attached invoice image and the user's message.\n\nUser Message: {user_text}\n"""

response = client.models.generate_content(
    model=MODEL_ID,
    # Notice we pass BOTH the text and the image in the contents list
    contents=[prompt, invoice_image],
    config=types.GenerateContentConfig(
        temperature=0.0,
        response_mime_type="application/json",
        response_schema=InvoiceExtraction,
    )
)

print("--- Multimodal Extraction Output ---")
extraction = InvoiceExtraction.model_validate_json(response.text)
print(extraction.model_dump_json(indent=2))

if extraction.is_contradictory:
    print("\n[APPLICATION LOGIC] Contradiction detected! Routing to human agent for review.")
else:
    print(f"\n[APPLICATION LOGIC] Automatically processing payment for ${extraction.extracted_amount}")


## Conclusion

By combining native multimodality (passing pixels directly to the model) with Structured Outputs (Pydantic schemas), we can build highly robust data pipelines that fail safely. Instead of guessing when faced with contradictory evidence, the model structures its uncertainty, allowing the application layer to trigger a human-in-the-loop escalation.